# Notebook 04: Dictionary Mechanics, Hash Tables, and ML Parameter Trees

This notebook covers Python dictionaries and their role in machine learning configuration management:
1. **Creation & Initialization:** Literals, `dict(zip(...))`, and dict comprehensions.
2. **Access & Mutation:** Square brackets vs `.get()` vs `.setdefault()`.
3. **Deletion Patterns:** `pop()`, `popitem()`, `del`, and `clear()`.
4. **Dictionary Merging:** `{**d1, **d2}` unpack and Python 3.9+ union operator `d1 | d2`.
5. **Memory Semantics:** Shallow copy (`.copy()`) vs Deepcopy (`copy.deepcopy()`).


In [ ]:
import copy
import json
from pathlib import Path

DATA_PATH = Path("../data/ml_hyperparameters.json")
with DATA_PATH.open("r", encoding="utf-8") as f:
    config = json.load(f)

print("Loaded ML hyperparameters config. Top-level keys:", list(config.keys()))

## 1. Dictionary Initialization Methods


In [ ]:
# 1. Dict comprehension
powers = {str(i): i**2 for i in range(5)}
assert powers == {"0": 0, "1": 1, "2": 4, "3": 9, "4": 16}

# 2. From zipped sequences
keys = ["learning_rate", "optimizer", "batch_size"]
vals = [0.001, "AdamW", 32]
zipped_dict = dict(zip(keys, vals, strict=True))
assert zipped_dict["optimizer"] == "AdamW"

print("Dictionary initialization verified!")

## 2. Value Retrieval: Brackets vs. `.get()` vs. `.setdefault()`


In [ ]:
params = {"learning_rate": 0.1, "optimizer": "Adam", "metric": "Accuracy"}

# 1. Bracket lookup: raises KeyError if absent
assert params["learning_rate"] == 0.1
try:
    _ = params["weight_decay"]
    raise AssertionError("Should have raised KeyError")
except KeyError:
    pass

# 2. .get() lookup: safe with optional default
assert params.get("optimizer") == "Adam"
assert params.get("weight_decay") is None
assert params.get("weight_decay", 1e-4) == 1e-4

# 3. .setdefault(): returns existing value, or inserts default if absent
val_lr = params.setdefault("learning_rate", 0.001)
assert val_lr == 0.1  # existing was preserved

val_wd = params.setdefault("weight_decay", 1e-4)
assert val_wd == 1e-4  # inserted new default
assert "weight_decay" in params

print("Retrieval semantics verified!")

## 3. Removal & Item Deletion: `del`, `pop`, `popitem`, `clear`


In [ ]:
d = {"a": 1, "b": 2, "c": 3, "d": 4}

# del statement
del d["a"]
assert "a" not in d

# pop(): removes and returns value
val_b = d.pop("b")
assert val_b == 2

# popitem(): removes and returns last inserted (key, value) tuple
k_last, v_last = d.popitem()
assert (k_last, v_last) == ("d", 4)

# clear(): empties container in place
d.clear()
assert d == {}

print("Deletion patterns verified!")

## 4. Dictionary Merging and Comprehensions


In [ ]:
base_cfg = {"model": "CNN", "lr": 0.01, "device": "cpu"}
override_cfg = {"lr": 0.001, "device": "cuda", "batch_size": 64}

# Unpack merge: {**d1, **d2}
merged_unpack = {**base_cfg, **override_cfg}

# Pipe operator merge (Python 3.9+)
merged_pipe = base_cfg | override_cfg

assert merged_unpack == merged_pipe
assert merged_pipe["lr"] == 0.001
assert merged_pipe["model"] == "CNN"
assert merged_pipe["batch_size"] == 64

# Filter empty / None values via comprehension
dirty_dict = {"lr": 0.01, "momentum": None, "decay": 1e-4, "notes": None}
cleaned_dict = {k: v for k, v in dirty_dict.items() if v is not None}
assert cleaned_dict == {"lr": 0.01, "decay": 1e-4}

print("Merging and comprehension filtering verified!")

## 5. Memory Hazards: Shallow Copy vs. Deep Copy in ML Configurations


In [ ]:
# Shallow copy: top-level copied, nested references shared
model_cfg = {"architecture": "MLP", "hidden_layers": [256, 128]}

shallow_cfg = model_cfg.copy()
shallow_cfg["architecture"] = "Transformer"
shallow_cfg["hidden_layers"].append(64)

# Danger: hidden_layers mutated across BOTH dictionaries!
assert model_cfg["architecture"] == "MLP"
assert model_cfg["hidden_layers"] == [256, 128, 64]  # Unintended side-effect!

# Deep copy: independent recursive clones
deep_model_cfg = {"architecture": "MLP", "hidden_layers": [256, 128]}

deep_cfg = copy.deepcopy(deep_model_cfg)
deep_cfg["hidden_layers"].append(64)

# Isolated: original remains intact
assert deep_model_cfg["hidden_layers"] == [256, 128]
assert deep_cfg["hidden_layers"] == [256, 128, 64]

print("Copy semantics verified! Deepcopy prevents hyperparameter corruption.")